# TRPO: constrain how far a policy can move

Learn **Trust Region Policy Optimization (TRPO)** from basic Gymnasium, NumPy, and PyTorch components on LunarLander. Read [n-step Actor-Critic](05_actor_critic_n_step.ipynb) first: we reuse on-policy rollouts and a value baseline, then replace the actor's gradient-descent optimizer with a constrained update.

Our objective is the expected discounted return
$$J(\theta)=\mathbb E_{\pi_\theta}\left[\sum_{t=0}^{T-1}\gamma^t r_{t+1}\right],$$
where $\theta$ denotes actor parameters, $\pi_\theta(a\mid s)$ the action distribution, $s_t$ a state, $a_t$ an action, $r_{t+1}$ its reward, $\gamma$ the discount factor, and $T$ the episode length.

A large ordinary policy-gradient step can destroy a useful policy. TRPO instead optimizes a local surrogate while limiting policy change:
$$\max_\theta L(\theta)=\frac1N\sum_t
\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)}\hat A_t,
\qquad
\bar D_{KL}(\theta)=\frac1N\sum_t D_{KL}\big(\pi_{\theta_{old}}(\cdot\mid s_t)\|\pi_\theta(\cdot\mid s_t)\big)\le\delta.$$
Here $N$ is the rollout size, $\theta_{old}$ the frozen collecting policy, $\hat A_t$ an estimated advantage, and $\delta$ the KL budget. The ratio reweights actions sampled by the old policy. We average over sampled states and sum the categorical KL over **all actions** at each state.

This is a practical approximation to the procedure in [Schulman et al., Trust Region Policy Optimization (2015)](https://arxiv.org/abs/1502.05477). A sampled mean-KL constraint does not guarantee improvement in true return or bound the KL at every state.

## 1. Set up a small CPU experiment

Use $\gamma=0.99$ and a named KL budget `MAX_KL`. The actor has no Adam learning rate: its step size comes from the constraint. `VALUE_LEARNING_RATE` controls only the critic. Sampling the categorical policy supplies exploration.

LunarLander uses eight observation features and four discrete actions; the networks infer these dimensions from the environment. Install its optional physics dependency with `pip install "gymnasium[box2d]>=1.0,<2"` (install `swig` first if the Box2D build requires it). See the [Gymnasium LunarLander documentation](https://gymnasium.farama.org/environments/box2d/lunar_lander/). The 30,000-step run is a short demonstration; learning a reliable landing policy may require a larger `TOTAL_TIMESTEPS` budget.


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical, kl_divergence
from torch.nn.utils import parameters_to_vector, vector_to_parameters

SEED = 7
ENV_ID = "LunarLander-v3"
TOTAL_TIMESTEPS = 50_000
N_STEPS = 1024
HIDDEN_SIZE = 32
GAMMA = 0.99
GAE_LAMBDA = 0.95
MAX_KL = 0.01
DAMPING = 0.1
CG_STEPS = 10
CG_TOLERANCE = 1e-10
BACKTRACK_STEPS = 10
BACKTRACK_COEFFICIENT = 0.5
ACCEPT_RATIO = 0.1
VALUE_LEARNING_RATE = 1e-3
VALUE_EPOCHS = 10
EPS = 1e-8
EVAL_EPISODES = 5

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)
device = torch.device("cpu")

env = gym.make(ENV_ID, render_mode="human") # set render_mode=None for faster training
env.metadata["render_fps"] = 1000

observation_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

## 2. Build separate actor and critic networks

The actor maps $s$ to logits $z_\theta(s)$, with $\pi_\theta(a\mid s)=\operatorname{softmax}(z_\theta(s))_a$. The critic $V_\phi(s)$ estimates discounted return with its own parameters $\phi$. Separate networks ensure critic fitting cannot change the actor after its KL constraint is checked. `select_action` samples during training and takes the most probable action during evaluation.

In [ ]:
def make_network(output_dim):
    return nn.Sequential(
        nn.Linear(observation_dim, HIDDEN_SIZE),
        nn.Tanh(),
        nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
        nn.Tanh(),
        nn.Linear(HIDDEN_SIZE, output_dim),
    ).to(device)


actor = make_network(action_dim)
critic = make_network(1)
actor_parameters = list(actor.parameters())
critic_optimizer = torch.optim.Adam(
    critic.parameters(), lr=VALUE_LEARNING_RATE
)


def action_distribution(observations):
    return Categorical(logits=actor(observations))


@torch.no_grad()
def select_action(observation, deterministic=False):
    states = torch.as_tensor(
        observation, dtype=torch.float32, device=device
    ).unsqueeze(0)
    distribution = action_distribution(states)
    action = (
        distribution.probs.argmax(dim=-1)
        if deterministic else distribution.sample()
    )
    return int(action.item())

## 3. Estimate advantages with two boundary masks

We use generalized advantage estimation (GAE):
$$d_t=r_{t+1}+\gamma(1-\mathrm{terminated}_t)V_\phi(s_{t+1})-V_\phi(s_t),$$
$$\hat A_t=d_t+\gamma\lambda(1-\mathrm{episode\_end}_t)\hat A_{t+1},
\qquad y_t=\hat A_t+V_\phi(s_t).$$
Here $d_t$ is the TD residual, $\lambda$ (`GAE_LAMBDA`) mixes multi-step estimates, and $y_t$ is the critic target. The recursion starts at zero beyond the rollout. A true termination removes the bootstrap; a time-limit truncation retains the value of the final observation. **Both** end the trace so it never crosses an environment reset. An ordinary rollout cutoff still bootstraps.

Compute targets without gradients, then standardize advantages for the actor. Keep unnormalized advantages for critic targets. Gradients must not flow through rewards, old policy quantities, advantages, or bootstrap values.

In [ ]:
@torch.no_grad()
def advantage_targets(rewards, values, next_values, terminated, episode_ends):
    residuals = rewards + GAMMA * (1.0 - terminated) * next_values - values
    advantages = torch.empty_like(rewards)
    running_advantage = torch.zeros((), device=device)
    for t in reversed(range(len(rewards))):
        running_advantage = (
            residuals[t]
            + GAMMA * GAE_LAMBDA * (1.0 - episode_ends[t]) * running_advantage
        )
        advantages[t] = running_advantage
    targets = advantages + values
    normalized = (advantages - advantages.mean()) / (
        advantages.std(unbiased=False) + EPS
    )
    return normalized, targets

## 4. Solve for a natural-gradient direction

At $\theta_{old}$, let $g=\nabla_\theta L$ and $H=\nabla_\theta^2\bar D_{KL}$. The local problem uses $g^T\Delta\theta$ for improvement and $\tfrac12\Delta\theta^TH\Delta\theta$ for KL. We solve $(H+\eta I)x=g$, where $\eta$ is `DAMPING` and $I$ the identity, using conjugate gradients instead of constructing or inverting a dense Hessian.

For a vector $v$, automatic differentiation computes $Hv=\nabla_\theta[(\nabla_\theta\bar D_{KL})^Tv]$. The first derivative needs `create_graph=True` so it can be differentiated again. The helper below flattens derivatives; `conjugate_gradient` tracks the residual $r=g-(H+\eta I)x$ and stops when its squared norm is small. Nonpositive or nonfinite curvature stops the solve defensively.

In [ ]:
def flat_gradient(scalar, create_graph=False):
    gradients = torch.autograd.grad(
        scalar, actor_parameters, create_graph=create_graph
    )
    return torch.cat([gradient.reshape(-1) for gradient in gradients])


def conjugate_gradient(matrix_vector_product, b):
    x = torch.zeros_like(b)
    residual = b.clone()
    direction = residual.clone()
    residual_squared = torch.dot(residual, residual)
    for _ in range(CG_STEPS):
        if residual_squared <= CG_TOLERANCE:
            break
        product = matrix_vector_product(direction)
        curvature = torch.dot(direction, product)
        if not torch.isfinite(curvature) or curvature <= 0:
            break
        alpha = residual_squared / curvature
        x = x + alpha * direction
        residual = residual - alpha * product
        next_squared = torch.dot(residual, residual)
        direction = residual + (next_squared / residual_squared) * direction
        residual_squared = next_squared
    return x

## 5. Scale the step and backtrack against the actual KL

Scale the direction as
$$\Delta\theta=\sqrt{\frac{2\delta}{x^T(H+\eta I)x}}\,x.$$
This uses damped curvature, making the quadratic estimate more conservative. The approximation can still be inaccurate. Try $\theta_{old}+\alpha\Delta\theta$ with $\alpha=1,\beta,\beta^2,\ldots$ and $\beta=$ `BACKTRACK_COEFFICIENT`.

Accept only finite candidates whose **actual** batch KL is at most $\delta$, whose surrogate improvement is positive, and whose improvement is at least `ACCEPT_RATIO` times the predicted gain $\alpha g^T\Delta\theta$. If every candidate fails, restore the old parameters exactly. Old logits and action log probabilities stay detached throughout the search. There is no actor optimizer step or clipped PPO objective.

In [ ]:
def update_actor(states, actions, advantages):
    with torch.no_grad():
        old_distribution = Categorical(logits=actor(states).detach().clone())
        old_log_probs = old_distribution.log_prob(actions)
    old_parameters = parameters_to_vector(actor_parameters).detach().clone()

    def objective_and_kl():
        distribution = action_distribution(states)
        ratio = torch.exp(distribution.log_prob(actions) - old_log_probs)
        objective = (ratio * advantages).mean()
        mean_kl = kl_divergence(old_distribution, distribution).mean()
        return objective, mean_kl

    objective, _ = objective_and_kl()
    old_objective = objective.detach()
    gradient = flat_gradient(objective).detach()

    def hessian_vector_product(vector):
        _, mean_kl = objective_and_kl()
        kl_gradient = flat_gradient(mean_kl, create_graph=True)
        product = flat_gradient(torch.dot(kl_gradient, vector)).detach()
        return product + DAMPING * vector

    direction = conjugate_gradient(hessian_vector_product, gradient)
    curvature = torch.dot(direction, hessian_vector_product(direction))
    result = {"kl": 0.0, "gain": 0.0, "step_fraction": 0.0}
    if not torch.isfinite(curvature) or curvature <= EPS:
        return result
    full_step = torch.sqrt(2.0 * MAX_KL / curvature) * direction
    expected_gain = torch.dot(gradient, full_step)
    if not torch.isfinite(expected_gain) or expected_gain <= 0:
        return result

    accepted = False
    try:
        with torch.no_grad():
            for backtrack in range(BACKTRACK_STEPS):
                fraction = BACKTRACK_COEFFICIENT ** backtrack
                vector_to_parameters(
                    old_parameters + fraction * full_step, actor_parameters
                )
                candidate_objective, candidate_kl = objective_and_kl()
                gain = candidate_objective - old_objective
                if (
                    torch.isfinite(candidate_kl)
                    and torch.isfinite(gain)
                    and candidate_kl <= MAX_KL
                    and gain > 0
                    and gain >= ACCEPT_RATIO * fraction * expected_gain
                ):
                    accepted = True
                    result = {
                        "kl": candidate_kl.item(),
                        "gain": gain.item(),
                        "step_fraction": fraction,
                    }
                    break
    finally:
        if not accepted:
            with torch.no_grad():
                vector_to_parameters(old_parameters, actor_parameters)
    return result

## 6. Fit the critic to fixed targets

Minimize $L_V(\phi)=\tfrac1N\sum_t(V_\phi(s_t)-y_t)^2$ for `VALUE_EPOCHS` passes. `update` computes $y_t$ and $\hat A_t$ before either network changes. The actor uses one constrained update per rollout; only the critic uses repeated Adam steps. Both are trained on the current rollout, which is then discarded.

In [ ]:
def update_critic(rollout):
    observations, actions, rewards, next_observations, terminated, ends = zip(
        *rollout, strict=True
    )

    def tensor(data):
        return torch.as_tensor(np.asarray(data), dtype=torch.float32, device=device)

    states, next_states = tensor(observations), tensor(next_observations)
    actions = torch.as_tensor(actions, dtype=torch.int64, device=device)
    with torch.no_grad():
        values = critic(states).squeeze(-1)
        next_values = critic(next_states).squeeze(-1)
        advantages, targets = advantage_targets(
            tensor(rewards), values, next_values, tensor(terminated), tensor(ends)
        )
    metrics = update_actor(states, actions, advantages)
    for _ in range(VALUE_EPOCHS):
        value_loss = nn.functional.mse_loss(critic(states).squeeze(-1), targets)
        critic_optimizer.zero_grad()
        value_loss.backward()
        critic_optimizer.step()
    metrics["value_loss"] = value_loss.item()
    return metrics

## 7. Collect on-policy rollouts and train

For each transition $(s_t,a_t,r_{t+1},s_{t+1})$, record both termination and episode-boundary flags. Store the final observation **before** resetting. Keep the policy fixed for `N_STEPS` transitions, then update; the final shorter rollout is also consumed. Episode return is the undiscounted sum $R=\sum_t r_{t+1}$. Its accumulator persists across rollout boundaries.

In [ ]:
def train(total_timesteps):
    episode_returns, history, rollout = [], [], []
    episode_return = 0.0
    observation, _ = env.reset(seed=SEED)
    try:
        for step in range(1, total_timesteps + 1):
            action = select_action(observation)
            next_observation, reward, terminated, truncated, _ = env.step(action)
            episode_end = terminated or truncated
            rollout.append((
                observation.copy(), action, reward, next_observation.copy(),
                terminated, episode_end,
            ))
            episode_return += reward
            if episode_end:
                episode_returns.append(episode_return)
                episode_return = 0.0
                observation, _ = env.reset()
            else:
                observation = next_observation
            if len(rollout) == N_STEPS or step == total_timesteps:
                metrics = update_critic(rollout)
                history.append(metrics)
                rollout.clear()
                last_return = np.mean(episode_returns[-10:]) if episode_returns else 0
                print(
                    f"Step {step:5d} | episodes {len(episode_returns)}"
                    f" | last return {last_return:6.1f}"
                    f" | KL {metrics['kl']:.5f}"
                    f" | step fraction {metrics['step_fraction']:.3f}"
                )
    finally:
        env.close()
    return episode_returns, history


episode_returns, history = train(TOTAL_TIMESTEPS)

## 8. Inspect returns and constraint diagnostics

The moving average $\bar R_i=\tfrac1w\sum_{j=i-w+1}^i R_j$ uses a window $w$ of up to ten episodes. Compare it with the measured mean KL and its budget. A zero step fraction means the actor update was rejected or skipped; a fraction below one means backtracking reduced the step. Critic loss can rise as the policy visits new states, and satisfying the KL constraint does not ensure that episode returns rise on every update.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
axes[0].plot(episode_returns, alpha=0.35, label="episode return")
if episode_returns:
    window = min(10, len(episode_returns))
    moving_average = np.convolve(
        episode_returns, np.ones(window) / window, mode="valid"
    )
    axes[0].plot(
        np.arange(window - 1, len(episode_returns)), moving_average,
        label=f"{window}-episode mean",
    )
axes[0].set(title=f"TRPO on {ENV_ID}", xlabel="Episode", ylabel="Return")
axes[0].legend()
axes[1].plot([item["kl"] for item in history], label="measured KL")
axes[1].axhline(MAX_KL, color="red", linestyle="--", label="KL budget")
axes[1].set(title="Policy change", xlabel="Update", ylabel="Mean KL")
axes[1].legend()
axes[2].plot([item["step_fraction"] for item in history])
axes[2].set(title="Accepted step fraction", xlabel="Update", ylabel="Fraction")
axes[3].plot([item["value_loss"] for item in history])
axes[3].set(title="Critic fit", xlabel="Update", ylabel="Mean squared error")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 9. Evaluate the deterministic policy

Choose $a_t=\arg\max_a\pi_\theta(a\mid s_t)$ in a separate rendered environment and report the mean and standard deviation of undiscounted returns. This measures the modal policy, which differs from the stochastic training policy. `rgb_array` rendering displays a final frame inline and works without a desktop window.

After running the notebook, try a smaller `MAX_KL`, disable damping, or vary `GAE_LAMBDA`. Restart the kernel for each comparison so weights and random seeds are reset. How do returns, accepted step fractions, and critic loss change? Why can a policy satisfy the mean KL budget while still changing substantially at a rare state?

In [ ]:
env = gym.make(ENV_ID, render_mode="human")
env.metadata["render_fps"] = 30

evaluation_returns = []
try:
    for episode in range(EVAL_EPISODES):
        observation, _ = env.reset(seed=SEED + 100 + episode)
        episode_return = 0.0
        while True:
            action = select_action(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            if terminated or truncated:
                break
        evaluation_returns.append(episode_return)
        print(f"Episode {episode + 1}: return={episode_return:.1f}")
    final_frame = env.render()
finally:
    env.close()

print(
    f"Mean return: {np.mean(evaluation_returns):.1f} "
    f"+/- {np.std(evaluation_returns):.1f}"
)